## Propósito
Aplicar de manera auditada la regla comunicada por la profesora Liliana: revisar folio, alcaldía y después el resto de campos. Los folios `SD` no se equiparan automáticamente. **Esta primera versión detecta candidatos y no borra filas** hasta validar unidad de observación, especialmente en tablas de personas y vehículos.

## Identificar candidatos y crear bitácora

In [ ]:
from pathlib import Path
import pandas as pd
SALIDA=Path("salidas")
FUENTE="SSC"
archivos=sorted(SALIDA.glob(f"{FUENTE}_*_ETAPA1.csv"))
bitacora=[]
for archivo in archivos:
    df=pd.read_csv(archivo,dtype=str,low_memory=False)
    fol=next((c for c in df.columns if c.lower()=="folio"),None)
    alc=next((c for c in df.columns if c.lower() in ("alcaldia","alcaldia_catalogo","alcaldia_inicio")),None)
    if fol and alc:
        valor=df[fol].fillna("").str.strip().str.upper()
        candidato=valor.ne("") & valor.ne("SD") & df.duplicated(subset=[fol,alc],keep=False) & df.duplicated(keep=False)
    else:
        candidato=pd.Series(False,index=df.index)
    candidatos=df.loc[candidato].copy()
    candidatos.to_csv(SALIDA/f"{archivo.stem}_CANDIDATOS_ETAPA3.csv",index=False,encoding="utf-8-sig")
    bitacora.append({"archivo":archivo.name,"filas_antes":len(df),"filas_candidatas":len(candidatos),"filas_eliminadas":0,"estado":"pendiente de confirmar unidad de observación y criterio"})
# No se elimina ninguna fila automáticamente. Tras revisar cada caso,
# crear una lista auditada de índices a retirar y conservar copia de esos registros.
reporte=pd.DataFrame(bitacora)
display(reporte)
reporte.to_csv(SALIDA/f"{FUENTE}_bitacora_etapa3_pendiente.csv",index=False,encoding="utf-8-sig")
